if i say that first follow import the library
then choose the llm where need 
make state
build graph
make node 
if i making the node also define the work of that node very propely 
make edge 


In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal,Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage,HumanMessage

In [ ]:
genrator_llm=ChatOpenAI(model="gpt-4o")
evaluator_llm=ChatOpenAI(model="gpt-4o-mini")
optimizer_llm=ChatOpenAI(model="gpt-40")

In [ ]:
from pyndantic import BaseModel,Field
class tweetevaluation(BaseModel):
    evaluation:Literal["approved","needs_improvement"]=Field(...,description="final evalution result")
    feedback:str=Field(...,description="feedback for the tweet")

In [ ]:
structured_evaluator_llm=evaluator_llm.with_structured_output(tweetevaluation)

In [ ]:
class tweetstate(TypedDict):
    topic:str
    tweet:str
    evaluation:Literal["approved","needs_approvement"]
    feedback:str
    iteration:int
    max_iteration:int


In [ ]:
graph=StateGraph(TypedDict)

In [ ]:

def generate_tweet(state):
    messages = [
        SystemMessage(
            content="You are a funny and clever Twitter/X influencer."
        ),
        HumanMessage(
            content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english.
- This is version {state['iteration'] + 1}.
"""
        )
    ]

    response = generator_llm.invoke(messages)

    return {
        "tweet": response.content
    }

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

def evaluate_tweet(state: tweetstate):
    messages = [
        SystemMessage(
            content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."
        ),
        HumanMessage(
            content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality - Is this fresh, or have you seen it a hundred times before?
2. Humor - Did it genuinely make you smile, laugh, or chuckle?
3. Punchiness - Is it short, sharp, and scroll-stopping?
4. Virality Potential - Would people retweet or share it?
5. Format - Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Don't end with generic, throwaway, or deflating lines that weaken the humor (e.g., "Masterpieces of the auntie-uncle universe" or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"
- feedback: One paragraph explaining the strengths and weaknesses
"""
        )
    ]

    response = structured_evaluator_llm.invoke(messages)

    return {
        "evaluation": response.evaluation,"feedback":response.feedback
    }

# creating the node first with proper variable and function 
<!-- graph.add_node('generate',generate_tweet)
graph.add_node('evaluate',evaluate_tweet)
graph.add_node('optimizer',optimizer_tweet) -->


In [ ]:
graph.add_node('generate',generate_tweet)
graph.add_node('evaluate',evaluate_tweet)
graph.add_node('optimizer',optimizer_tweet)
